<div class="freebsdLab-IntroCard">
  <div class="freebsdLab-IntroBrand">FreeBSD Laboratory</div>
  <h1 class="freebsdLab-IntroTitle">Autonomous AI Agent Subsystem</h1>
  <p class="freebsdLab-IntroTagline">Governed local model inference in disposable, isolated runtimes.</p>
  <p class="freebsdLab-IntroCopy">This notebook demonstrates the FreeBSD Laboratory autonomous agent controller. Small instruction-tuned LLMs (such as Gemma 2B via <code>llama-cpp-python</code>) propose guest-level shell actions executed inside disposable FreeBSD runtimes while preserving the repository's strict trust boundaries.</p>
  <div class="freebsdLab-IntroNotice">
    <div class="freebsdLab-IntroNoticeIcon">i</div>
    <div>
      <div class="freebsdLab-IntroNoticeLabel">Architecture Invariant</div>
      <div class="freebsdLab-IntroNoticeText">The model has zero visibility into <code>runtime.sock</code>, ZFS datasets, SSH key paths, or host filesystem details. The unprivileged Agent Controller governs all execution, enforces output/timeout bounds, and emits hash-only evidence logs.</div>
    </div>
    <div class="freebsdLab-IntroNoticeMeta"><strong>Agent</strong>bhyve + jail</div>
  </div>
</div>

## 1. Architecture & Trust Hierarchy

```text
┌──────────────────────┐
│  Local LLM Engine    │
│  llama-cpp-python    │
│  GGUF model          │
└──────────┬───────────┘
           │ proposed action (COMMAND: | FINAL:)
           ▼
┌──────────────────────┐
│  Agent Controller    │ unprivileged
│  command policy      │ timeout & output bounds
│  evidence generation │ context token budget
└──────┬────────┬──────┘
       │        │
       ▼        ▼
 RuntimeClient  SSHTransport
 (runtime.sock) (freebsd@, per-runtime key)
       │        │
       ▼        ▼
 runtime-daemon Isolated Runtime (bhyve default, jail opt-in)
```

- **bhyve default**: Full hardware virtualization isolation against adversarial code.
- **Unprivileged SSH**: Reuses `SSHTransport` with `freebsd` user and per-runtime Ed25519 keys (no root login).
- **Bounded I/O**: Stream-drained Popen execution with 4 KiB head + 4 KiB tail rolling buffers.

## 2. Action Protocol & Parsing

The model operates under a minimal two-action response grammar:
- `COMMAND: <single shell action>` — propose an action to run in the guest
- `FINAL: <task result>` — declare the task complete with summary findings

In [ ]:
from freebsd_laboratory.agent.model import parse_action
from freebsd_laboratory.agent.types import Command, FinalAnswer

# Test parsing different model response formats
sample_responses = [
    "COMMAND: sysctl hw.model hw.ncpu",
    "   COMMAND:   ifconfig vnet0   ",
    "I have analyzed the system.\nFINAL: FreeBSD 15.1 running on bhyve with 4 cores.",
    "df -h /",
]

for resp in sample_responses:
    action = parse_action(resp)
    print(f"Raw input : {resp!r}")
    print(f"Parsed as : {type(action).__name__} -> {getattr(action, 'command', getattr(action, 'answer', ''))}")
    print("-" * 60)

## 3. Structural Policy & Command Authorization

The controller enforces structural bounds (command byte length, step limits, session deadline). Because runtimes are disposable and network-isolated, destructive actions (e.g. `rm -rf /`) are contained inside the guest rather than fragilely denylisted.

In [ ]:
from freebsd_laboratory.agent.policy import AgentPolicy

policy = AgentPolicy(max_command_bytes=256, max_steps=5, max_runtime_seconds=60)

commands_to_test = [
    ("uname -a", 0, 1.2),
    ("echo " + "x" * 300, 1, 2.5),       # Exceeds byte cap
    ("rm -rf /tmp/scratch", 2, 4.0),     # Destructive but allowed in disposable runtime
    ("   ", 3, 5.0),                     # Empty command
    ("sysctl kern.ostype", 5, 8.0),      # Step limit reached
    ("uptime", 4, 65.0),                 # Session deadline exceeded
]

for cmd, step, elapsed in commands_to_test:
    decision = policy.authorize(cmd, step=step, elapsed=elapsed)
    status = "AUTHORIZED" if decision.authorized else f"REJECTED ({decision.reason})"
    print(f"Step {step} (t={elapsed:4.1f}s) | {cmd[:30]:<30} -> {status}")

## 4. Bounded Execution & Concurrent Stream Draining

`bounded_exec()` prevents controller memory exhaustion by draining stdout and stderr concurrently with reader threads, keeping the first 4 KiB (head) and last 4 KiB (rolling tail).

In [ ]:
import sys
from freebsd_laboratory.agent.bounded_exec import bounded_exec

# Run a simulated command producing 50 KiB of output
code = (
    "import sys\n"
    "sys.stdout.write('START_MARKER\n' + 'DATA_LINE\n' * 5000 + 'END_MARKER\n')\n"
)

exit_code, stdout_out, stderr_out = bounded_exec(
    [sys.executable, "-c", code],
    timeout=5.0,
    head_limit=256,
    tail_limit=256,
)

print(f"Exit status   : {exit_code}")
print(f"Total bytes   : {stdout_out.total_bytes} bytes")
print(f"Truncated?    : {stdout_out.truncated}")
print(f"Head snippet  : {stdout_out.head[:50]!r}...")
print(f"Tail snippet  : ...{stdout_out.tail[-50:]!r}")

## 5. Hash-Only Evidence Generation

The agent persists execution events into a standalone append-only JSONL log. To ensure privacy and prevent secret leakage, only SHA-256 digests and byte counters are stored on disk.

In [ ]:
import json
import tempfile
from pathlib import Path
from freebsd_laboratory.agent.evidence import AgentEvidenceLog, make_command_event
from freebsd_laboratory.agent.types import BoundedOutput, Observation

with tempfile.TemporaryDirectory() as tmpdir:
    log_file = Path(tmpdir) / "agent_evidence.jsonl"
    log = AgentEvidenceLog(log_file)
    
    # Create a simulated observation
    obs = Observation(
        step=0,
        command="kenv | grep -i bhyve",
        exit_status=0,
        stdout=BoundedOutput(head=b"smbios.planar.maker=bhyve\n", tail=b"", total_bytes=26, truncated=False),
        stderr=BoundedOutput(head=b"", tail=b"", total_bytes=0, truncated=False),
        duration_ms=45,
    )
    
    event = make_command_event("demo-session-01", "freebsd-lab-a123", "bhyve", obs)
    log.emit(event)
    log.close()
    
    # Inspect the saved JSONL line
    line = log_file.read_text().strip()
    parsed = json.loads(line)
    print("Persisted JSONL Event:")
    print(json.dumps(parsed, indent=2))
    
    # Verify no raw command is stored
    assert "kenv" not in line
    assert "smbios" not in line
    print("\n[PASSED] Invariant verified: No raw command or output strings persisted in evidence log.")

## 6. End-to-End Agent Orchestration Simulation

We simulate an autonomous agent session where the model diagnoses system resources and reaches a final answer within policy bounds.

In [ ]:
import subprocess
from unittest.mock import MagicMock
from freebsd_laboratory.agent.controller import AgentController
from freebsd_laboratory.agent.policy import AgentPolicy
from freebsd_laboratory.agent.types import Command, FinalAnswer, Observation, RuntimeHandle

# Mock model simulating an agent investigating FreeBSD kernel metrics
class SimulatedAgentModel:
    def __init__(self):
        self.turn = 0
        
    def next_action(self, goal, observations):
        self.turn += 1
        if self.turn == 1:
            return Command("sysctl -n kern.ostype kern.osrelease hw.ncpu")
        elif self.turn == 2:
            return Command("vmstat -s | head -n 4")
        else:
            return FinalAnswer(
                "System Audit Complete: Verified FreeBSD host metrics and active CPU scheduler."
            )

# Mock runtime executing commands against local host / disposable environment
class SimulatedAgentRuntime:
    def create(self):
        return RuntimeHandle(
            runtime_name="freebsd-lab-demo",
            guest_ip="172.31.254.10",
            runtime_type="bhyve",
            private_key=Path("/tmp/mock_key"),
            known_hosts_file=Path("/tmp/mock_kh"),
        )
        
    def execute(self, handle, command):
        res = subprocess.run(["sh", "-c", command], capture_output=True)
        return Observation(
            step=0,
            command=command,
            exit_status=res.returncode,
            stdout=BoundedOutput(head=res.stdout[:512], tail=b"", total_bytes=len(res.stdout), truncated=False),
            stderr=BoundedOutput(head=res.stderr[:512], tail=b"", total_bytes=len(res.stderr), truncated=False),
            duration_ms=15,
        )
        
    def destroy(self, handle):
        pass

# Run the controller loop
model = SimulatedAgentModel()
runtime = SimulatedAgentRuntime()
policy = AgentPolicy(max_steps=6, max_runtime_seconds=30)
controller = AgentController(model=model, runtime=runtime, policy=policy)

goal = "Inspect FreeBSD kernel type, version, and CPU availability"
print(f"Starting Agent Loop with Goal: {goal!r}\n")
result = controller.run(goal)
print(f"\nAgent Completed Successfully!\nFinal Output: {result}")

## 7. CLI Usage Reference

In production, the autonomous agent is executed using the `freebsd-lab-agent` command:

```sh
# Run an autonomous agent inside a strong bhyve VM boundary (default):
freebsd-lab-agent "Analyze /var/log/messages and report any kernel panic indicators" \
    --model /models/gemma-2b-it.gguf \
    --mode bhyve \
    --max-steps 16 \
    --max-runtime 300 \
    --evidence-dir .freebsd-lab/agent-evidence

# Run in a lightweight VNET jail for fast controlled benchmarks:
freebsd-lab-agent "Check installed package versions and audit report" \
    --model /models/gemma-2b-it.gguf \
    --mode jail
```